# Original TabNet Baseline Model

* **Objective:** Establish a standard baseline performance using the original TabNet architecture without advanced missingness handling.
* **Key Libraries:**
  * `pytorch-tabnet`: Core framework for the TabNet architecture.
  * `optuna`: Framework for automated hyperparameter optimization.
* **Preprocessing Strategy:**
  * **Categorical variables:** Imputed with "Unknown" and label-encoded.
  * **Numerical variables:** Imputed with the median value.
  * **Data Split:** Train (2011-2014), Validation (2015), Retrain (2011-2015), Test (2016).

In [ ]:
# ==========================================
# [1] Required package installation & Setup
# ==========================================
!pip install pytorch-tabnet pytorch-widedeep optuna -q

import pandas as pd
import numpy as np
import torch
import io
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TabNet (original implementation)
from pytorch_tabnet.tab_model import TabNetRegressor

# ==========================================
# [1] Data loading and preprocessing
#     (standard TabNet baseline setting)
# ==========================================
def load_and_prep():
    """
    Loads dataset and applies basic preprocessing for the baseline TabNet.
    """
    # 1. Load dataset
    try:
        from google.colab import files
        print("=== [Google Colab] Please upload Dataset.csv ===")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[filename]))
    except ImportError:
        print("=== [Local Environment] Loading Dataset.csv ===")
        df = pd.read_csv('Dataset.csv')

    # 2. Basic preprocessing
    if 'Country Name' in df.columns:
        df = df.drop('Country Name', axis=1)

    cat_cols = ['Country Code', 'Continent']

    # Categorical missing values -> "Unknown"
    for col in cat_cols:
        df[col] = df[col].fillna("Unknown")

    target = 'Maternal Mortality Ratio'
    exclude = cat_cols + ['Year', target]
    num_cols = [c for c in df.columns if c not in exclude]

    # Numerical missing values -> Median imputation
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    # 3. Encoding and split setup
    categorical_dims = {}
    for col in cat_cols:
        l_enc = LabelEncoder()
        df[col] = l_enc.fit_transform(df[col].values)
        categorical_dims[col] = len(l_enc.classes_)

    features = [c for c in df.columns if c not in ['Year', target]]
    cat_idxs = [i for i, f in enumerate(features) if f in cat_cols]
    cat_dims = [categorical_dims[f] for f in features if f in cat_cols]

    # Temporal data splits
    # Phase 1: Model selection (Train 2011-2014 / Validation 2015)
    train_p1 = df[(df['Year'] >= 2011) & (df['Year'] <= 2014)]
    val_p1   = df[df['Year'] == 2015]

    # Phase 2: Final retraining on the pretest period (2011-2015)
    train_p2 = df[(df['Year'] >= 2011) & (df['Year'] <= 2015)]

    # Held-out test year
    test = df[df['Year'] == 2016]

    # Helper function to separate features and target
    def to_xy(data):
        return data[features].values, data[target].values.reshape(-1, 1)

    return (to_xy(train_p1), to_xy(val_p1), to_xy(train_p2), to_xy(test),
            cat_idxs, cat_dims, features)

# Prepare data arrays
(data_train_1, data_val_1, data_train_2, data_test,
 cat_idxs, cat_dims, feature_names) = load_and_prep()

X_train_1, y_train_1 = data_train_1
X_val_1, y_val_1     = data_val_1
X_train_2, y_train_2 = data_train_2
X_test, y_test       = data_test

## 1. Hyperparameter Optimization

* **Optuna Search:**
  * Defines a specific search space for TabNet parameters including layer widths (`n_d`, `n_a`), sequential steps (`n_steps`), and learning rate.
  * Evaluates configurations using the 2015 validation set to find the optimal architecture before final retraining.

In [ ]:
## 1. Hyperparameter Optimization

* **Optuna Search:**
  * Defines a specific search space for TabNet parameters including layer widths (`n_d`, `n_a`), sequential steps (`n_steps`), and learning rate.
  * Evaluates configurations using the 2015 validation set to find the optimal architecture before final retraining.

In [ ]:
# ==========================================
# [2] Optuna-based hyperparameter optimization
# ==========================================
def objective(trial):
    # Search space definition
    param = {
        'n_d': trial.suggest_int('n_d', 8, 64, step=8),          # width of decision layers
        'n_a': trial.suggest_int('n_a', 8, 64, step=8),          # width of attention layers
        'n_steps': trial.suggest_int('n_steps', 3, 10),          # number of sequential decision steps
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),         # relaxation parameter
        'n_independent': trial.suggest_int('n_independent', 1, 5),
        'n_shared': trial.suggest_int('n_shared', 1, 5),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-4, 1e-2, log=True),
        'optimizer_params': {'lr': trial.suggest_float('lr', 1e-3, 1e-1, log=True)},
    }

    # Baseline TabNet model
    clf = TabNetRegressor(
        cat_idxs=cat_idxs,
        cat_dims=cat_dims,
        cat_emb_dim=1,
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        verbose=0,
        **param
    )

    # Train on the model-selection split
    clf.fit(
        X_train=X_train_1, y_train=y_train_1,
        eval_set=[(X_val_1, y_val_1)],
        eval_name=['valid'],
        eval_metric=['mae'],
        max_epochs=100,
        patience=15,
        batch_size=256,
        virtual_batch_size=128,
        drop_last=False
    )

    # Return the best validation score
    return clf.best_cost

print("\n>>> [Step 1] Starting Optuna hyperparameter optimization (20 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

print("\n>>> [Result] Best hyperparameters")
best_params = study.best_trial.params
print(best_params)

## 2. Model Retraining & Bootstrap Function

* **Retraining:** * The optimized TabNet model is retrained on the entire pretest dataset (2011-2015).
* **Bootstrap Evaluation:** * Computes stable 95% Confidence Intervals for error metrics (MAE, RMSE, R2) through 1,000 resampling iterations.

In [ ]:
# ==========================================
# [3] Retraining on the full pretest period
# ==========================================
print("\n>>> [Step 2] Retraining the optimized TabNet baseline on 2011-2015...")

# Parse optimized parameters
final_params = best_params.copy()
lr = final_params.pop('lr')

# Initialize final model with the best hyperparameter configuration
clf_final = TabNetRegressor(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,
    optimizer_params={'lr': lr},
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    verbose=10,
    **final_params
)

# Train on the combined 2011-2015 dataset
clf_final.fit(
    X_train=X_train_2, y_train=y_train_2,
    eval_set=None,
    max_epochs=150,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

# ==========================================
# Bootstrap Confidence Interval Function
# ==========================================
def calculate_bootstrap_ci(y_true, y_pred, n_bootstraps=1000, ci=95, random_state=42):
    """
    Calculate the mean performance and confidence intervals using bootstrapping.

    Parameters:
    - y_true: Ground truth target values (1D array)
    - y_pred: Model predictions (1D array)
    - n_bootstraps: Number of resampling iterations (default 1000)
    - ci: Confidence interval percentage (default 95%)
    - random_state: Seed for reproducibility

    Returns:
    - Dictionary mapping metric names to 'Mean (Lower_Bound - Upper_Bound)' formatted strings
    """
    # Standardize array shapes
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    n_samples = len(y_true)
    rng = np.random.RandomState(random_state)

    # Lists to store metric values per iteration
    boot_mae, boot_rmse, boot_r2 = [], [], []

    for _ in range(n_bootstraps):
        # 1. Random sampling with replacement
        indices = rng.randint(0, n_samples, n_samples)

        # 2. Create virtual test set
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # 3. Compute and store metrics
        boot_mae.append(mean_absolute_error(y_true_boot, y_pred_boot))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        boot_r2.append(r2_score(y_true_boot, y_pred_boot))

    # Calculate percentiles for lower and upper bounds
    lower_p = (100 - ci) / 2.0
    upper_p = 100 - lower_p

    results = {}
    metrics = {'MAE': boot_mae, 'RMSE': boot_rmse, 'R2': boot_r2}

    for name, values in metrics.items():
        # Use the mean of the bootstrap distribution
        mean_val = np.mean(values)
        lower_val = np.percentile(values, lower_p)
        upper_val = np.percentile(values, upper_p)

        results[name] = f"{mean_val:.4f} ({lower_val:.4f} - {upper_val:.4f})"

    return results

## 3. Final Evaluation and Visualizations

* **Performance Assessment:**
  * Outputs the final test metrics and their 95% Bootstrap Confidence Intervals for the unseen 2016 dataset.
* **Explainability Plots:**
  * **Feature Importance:** Highlights the top 10 most influential features driving the baseline TabNet model.
  * **Attention Masks:** Visualizes TabNet's inherent interpretability by showing which features are focused on during different sequential decision steps.

In [ ]:
# ==========================================
# [4] Final evaluation and visualization
# ==========================================
preds = clf_final.predict(X_test)

# Print deterministic metrics
print("\n=== [Step 3] Held-out Test Results (Year 2016) ===")
print(f"MAE : {mean_absolute_error(y_test, preds):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.4f}")
print(f"R2  : {r2_score(y_test, preds):.4f}")

# Calculate and print Bootstrap Confidence Intervals
bootstrap_results = calculate_bootstrap_ci(y_test, preds, n_bootstraps=1000)
print("\n=== [95% Bootstrap Confidence Intervals (1000 iter)] ===")
print(f"MAE  : {bootstrap_results['MAE']}")
print(f"RMSE : {bootstrap_results['RMSE']}")
print(f"R2   : {bootstrap_results['R2']}")

# ------------------------------------------
# Feature importance plot
# ------------------------------------------
plt.figure(figsize=(10, 5))
importances = pd.Series(clf_final.feature_importances_, index=feature_names)

# Plot top 10 features horizontally
importances.nlargest(10).sort_values().plot(kind='barh', color='#6c5ce7', edgecolor='0.25')
plt.title('TabNet Baseline Feature Importance', fontweight='bold')
plt.xlabel("Importance")

# Clean aesthetics
ax = plt.gca()
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ------------------------------------------
# Attention masks visualization
# ------------------------------------------
# Extract global attention masks built into TabNet
explain_matrix, masks = clf_final.explain(X_test)

# Visualize the masks for the first 3 decision steps for the first 15 samples
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    # Select the first 15 samples for clear visualization
    sns.heatmap(masks[i][:15], ax=axs[i], cbar=False, cmap='viridis')
    axs[i].set_title(f"Decision-Step Mask {i} (Top 15 Samples)", fontweight='bold')
    axs[i].set_xlabel("Features")
    axs[i].set_ylabel("Samples")

plt.suptitle("TabNet Baseline Attention Masks", y=1.05, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()